In [2]:
import os
import shutil

# Defining the directories for the two parts and the destination folder
part1_dir = r'C:\users\sanju\Downloads\archive\images\HAM10000_images_part_1'  # extracted path
part2_dir = r'C:\users\sanju\Downloads\archive\images\HAM10000_images_part_2'  # extracted path
merged_dir = r'C:\Users\Sanju\OneDrive\Desktop\final' # Directory where all images are placed

# Creating the merged directory if it doesn't exist
os.makedirs(merged_dir, exist_ok=True)

# Function to move images from one part to the merged directory
def move_images(src_dir, dest_dir):
    for filename in os.listdir(src_dir):
        src_path = os.path.join(src_dir, filename)
        dest_path = os.path.join(dest_dir, filename)
        
        if os.path.isfile(src_path):  # Checking if it's a file
            shutil.copy(src_path, dest_path)  # Copying the image to the destination

# Moviing images from part 1 and part 2
move_images(part1_dir, merged_dir)
move_images(part2_dir, merged_dir)

print(f"All images have been merged into {merged_dir}")


All images have been merged into C:\Users\Sanju\OneDrive\Desktop\final


In [35]:
import pandas as pd

# Path to the metadata file
metadata_file = r'C:\Users\Sanju\Downloads\archive\HAM10000_metadata.csv'  #  file path

# Load the metadata file
metadata = pd.read_csv(metadata_file)

# Assuming the label column is named 'dx'
unique_labels = metadata['dx'].unique()

# Print unique labels
print("Unique labels:", unique_labels)


Unique labels: ['bkl' 'nv' 'df' 'mel' 'vasc' 'bcc' 'akiec']


In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import shutil

# Paths
base_dir = r'C:\Users\Sanju\OneDrive\Desktop\final'  # Directory containing images
csv_path = r'C:\Users\Sanju\Downloads\archive\HAM10000_metadata.csv'  # Path to the metadata CSV
output_dir = r'C:\Users\Sanju\OneDrive\Desktop\finally'  # Output directory for sorted images

# Loading metadata
labels_df = pd.read_csv(csv_path)

# Check that required columns exist
if 'image_id' not in labels_df.columns or 'dx' not in labels_df.columns:
    raise ValueError("The CSV file must contain 'image_id' and 'dx' columns.")

# Adding a new column for filenames (assuming images are named as '<image_id>.jpg')
labels_df['filename'] = labels_df['image_id'] + '.jpg'

# Get unique labels (from 'dx' column)
unique_labels = labels_df['dx'].unique()

# Creating directories for train, validation, and test sets
for split in ['train', 'validation', 'test']:
    for label in unique_labels:
        os.makedirs(os.path.join(output_dir, split, label), exist_ok=True)

# Spliting the data
train_df, test_df = train_test_split(labels_df, test_size=0.2, stratify=labels_df['dx'], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.25, stratify=train_df['dx'], random_state=42)  # 0.25 x 0.8 = 0.2

# Function to copy files to the respective directories
def copy_files(df, split):
    for _, row in df.iterrows():
        src = os.path.join(base_dir, row['filename'])
        dest = os.path.join(output_dir, split, row['dx'], row['filename'])
        if os.path.exists(src):
            shutil.copy(src, dest)
        else:
            print(f"File not found: {src}")

# Copying files to train, validation, and test directories
copy_files(train_df, 'train')
copy_files(val_df, 'validation')
copy_files(test_df, 'test')

print("Images have been successfully split into train, validation, and test sets.")


Images have been successfully split into train, validation, and test sets.


In [1]:
import sys
print(sys.version)

import tensorflow as tf
print(tf.__version__)

3.12.0 (tags/v3.12.0:0fb18b0, Oct  2 2023, 13:03:39) [MSC v.1935 64 bit (AMD64)]
2.21.0


In [2]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Defining paths to train, validation, and test directories
train_dir = r'C:\Users\Sanju\OneDrive\Desktop\finally\train'
valid_dir = r'C:\Users\Sanju\OneDrive\Desktop\finally\validation'
test_dir = r'C:\Users\Sanju\OneDrive\Desktop\finally\test'

# Defining ImageDataGenerators for real-time data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,  # Normalize pixel values to [0, 1]
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

valid_datagen = ImageDataGenerator(rescale=1./255)  # No augmentation for validation

test_datagen = ImageDataGenerator(rescale=1./255)  # No augmentation for test

# Loading data in batches from the directories
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),  # Resizing images to 224x224 
    batch_size=32,
    class_mode='categorical',  
    shuffle=True
)

valid_generator = valid_datagen.flow_from_directory( 
    valid_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False  # no shuffle validation data
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)


Found 6009 images belonging to 7 classes.
Found 2003 images belonging to 7 classes.
Found 2003 images belonging to 7 classes.


In [3]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D,
    Flatten, Dense, Dropout,
    BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Build CNN Model
model = Sequential([

    Input(shape=(224,224,3)),

    Conv2D(32, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Conv2D(256, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Flatten(),

    Dense(256, activation='relu'),
    Dropout(0.5),

    Dense(128, activation='relu'),
    Dropout(0.3),

    Dense(7, activation='softmax')   # 7 skin lesion classes
])

# Compile Model
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Early Stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Train Model
history = model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=20,
    callbacks=[early_stop]
)

# Evaluate Model
test_loss, test_acc = model.evaluate(test_generator)

print("Test Accuracy:", test_acc * 100)

Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 867s 5s/step - accuracy: 0.5981 - loss: 1.3800 - val_accuracy: 0.6695 - val_loss: 2.2252
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 692s 4s/step - accuracy: 0.6350 - loss: 1.0956 - val_accuracy: 0.6715 - val_loss: 1.6165
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 655s 3s/step - accuracy: 0.6402 - loss: 1.0818 - val_accuracy: 0.6795 - val_loss: 0.9324
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 581s 3s/step - accuracy: 0.6500 - loss: 1.0493 - val_accuracy: 0.6960 - val_loss: 0.8548
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 586s 3s/step - accuracy: 0.6439 - loss: 1.0367 - val_accuracy: 0.6895 - val_loss: 0.8834
Epoch 6/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 671s 4s/step - accuracy: 0.6640 - loss: 0.9878 - val_accuracy: 0.6900 - val_loss: 0.8304
Epoch 7/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 533s 3s/step - accuracy: 0.6535 - loss: 1.0065 - val_accuracy: 0.6955 - val_loss: 0.8561
Epoch 8/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 588s 3s/step - accuracy: 0.6595 - loss: 0.9836 - val_accu

In [4]:
model.save("skin_lesion_model1.h5")

print("Model saved successfully!")

Model saved successfully!


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Loading pretrained MobileNetV2
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# Freezing pretrained layers
base_model.trainable = False

# Building the model
model = Sequential([
    base_model,

    GlobalAveragePooling2D(),

    Dense(256, activation='relu'),

    Dropout(0.5),

    Dense(7, activation='softmax')
])

# Compiling
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Early stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Train
history = model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=10,
    callbacks=[early_stop]
)

# Evaluate
test_loss, test_acc = model.evaluate(test_generator)

print("Test Accuracy:", test_acc * 100)